In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_dog.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_dog.jpg
wget --no-verbose --output-document=image_me.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_me.jpg

# Lecture 11 - Segmentation

Semantic segmentation, the checkerboard artefacts of transposed convolutions, instance segmentation and keypoints with Mask/Keypoint R-CNN, and interactive segmentation with SAM.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
def load_rgb(filename, scale=1.0):
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (0, 0), fx=scale, fy=scale) if scale != 1.0 else image

def to_tensor(image):
    return torch.from_numpy(image).permute(2, 0, 1).float().div(255).to(device)

dog = load_rgb('image_dog.jpg')
me = load_rgb('image_me.jpg')
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for ax, im in zip(axs, [dog, me]):
    ax.imshow(im)
    ax.axis('off')
plt.show()

## Semantic segmentation

A fully convolutional DeepLabV3 (ResNet-50 backbone, dilated convolutions) predicts one of the 21 Pascal VOC classes for every pixel. Note that the output is computed at 1/8 of the input resolution and upsampled bilinearly.

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
weights = DeepLabV3_ResNet50_Weights.DEFAULT
segmenter = deeplabv3_resnet50(weights=weights).to(device).eval()
voc_classes = weights.meta['categories']
normalize = torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
palette = (plt.cm.tab20(np.arange(len(voc_classes)) % 20)[:, :3] * 255).astype(np.uint8)
palette[0] = 0  # background

def segment(image):
    with torch.no_grad():
        logits = segmenter(normalize(to_tensor(image))[None])['out'][0]
    return logits.softmax(0).cpu().numpy()

fig, axs = plt.subplots(2, 3, figsize=(21, 10))
for row, image in zip(axs, [dog, me]):
    probs = segment(image)
    labels = probs.argmax(0)
    row[0].imshow(image)
    row[0].set_title('Image')
    row[1].imshow((0.4 * image + 0.6 * palette[labels]).astype(np.uint8))
    row[1].set_title('Predicted classes: ' + ', '.join(voc_classes[c] for c in np.unique(labels)))
    row[2].imshow(probs.max(0), cmap='viridis', vmin=0, vmax=1)
    row[2].set_title('Confidence (max. class probability)')
    for ax in row:
        ax.axis('off')
savefig('segmentation_semantic')
plt.show()

## Transposed convolutions and checkerboard artefacts

A transposed convolution with stride $s$ "paints" a copy of its $k \times k$ kernel for every input pixel, $s$ pixels apart. If $k$ is not divisible by $s$, the copies overlap unevenly: some output pixels receive more contributions than others. Even for a constant input and a constant kernel the output is a checkerboard ([Odena et al., 2016](https://distill.pub/2016/deconv-checkerboard/)). Upsampling first (nearest neighbour) and then convolving avoids this.

In [ ]:
x = torch.ones(1, 1, 8, 8)
outputs = {
    'transposed conv, k=3, s=2': F.conv_transpose2d(x, torch.ones(1, 1, 3, 3), stride=2),
    'transposed conv, k=4, s=2': F.conv_transpose2d(x, torch.ones(1, 1, 4, 4), stride=2),
    'nearest upsampling + conv, k=3': F.conv2d(F.interpolate(x, scale_factor=2, mode='nearest'), torch.ones(1, 1, 3, 3), padding=1),
}
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, y) in zip(axs, outputs.items()):
    y = y[0, 0].numpy()
    ax.imshow(y, cmap='gray', vmin=0)
    for (i, j), v in np.ndenumerate(y):
        ax.text(j, i, f'{v:.0f}', ha='center', va='center', color='red', fontsize=7)
    ax.set_title(f'{name}\n(input: 8x8 ones, output {y.shape[0]}x{y.shape[1]})')
    ax.axis('off')
savefig('segmentation_checkerboard')
plt.show()

# the same effect on an image: upsample 4x with two stride-2 transposed convolutions with a (normalised) box kernel
small = torch.from_numpy(cv2.resize(dog, (160, 107))).permute(2, 0, 1).float()[:, None] / 255
k3 = torch.ones(1, 1, 3, 3) / 2.25   # 2.25 = average number of overlapping kernel copies (9 / 4)
up = F.conv_transpose2d(F.conv_transpose2d(small, k3, stride=2), k3, stride=2)
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
axs[0].imshow(small[:, 0].permute(1, 2, 0)[30:70, 50:100], interpolation='nearest')
axs[0].set_title('Input (crop)')
axs[1].imshow(up[:, 0].permute(1, 2, 0).clamp(0, 1)[120:280, 200:400], interpolation='nearest')
axs[1].set_title('Two transposed convolutions k=3, s=2 (crop)')
for ax in axs:
    ax.axis('off')
plt.show()

## Instance segmentation and keypoints

Mask R-CNN adds a small fully convolutional mask head to Faster R-CNN: for every detected box it predicts a 28x28 mask inside the box (after RoIAlign), which is resized to the box. Keypoint R-CNN predicts one heatmap per keypoint instead of a mask.

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection import keypointrcnn_resnet50_fpn, KeypointRCNN_ResNet50_FPN_Weights
mask_weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
maskrcnn = maskrcnn_resnet50_fpn_v2(weights=mask_weights).to(device).eval()
coco_names = mask_weights.meta['categories']
kp_weights = KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
keypointrcnn = keypointrcnn_resnet50_fpn(weights=kp_weights).to(device).eval()
keypoint_names = kp_weights.meta['keypoint_names']
skeleton = [(15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6), (5, 7), (6, 8), (7, 9), (8, 10),
            (1, 2), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6)]

def show_instances(ax, image, out, score_thresh=0.6):
    overlay = image.astype(np.float32)
    for i, (box, label, score) in enumerate(zip(out['boxes'], out['labels'], out['scores'])):
        if score < score_thresh:
            continue
        color = np.array(plt.cm.tab10(i % 10)[:3])
        mask = out['masks'][i, 0] > 0.5
        overlay[mask] = 0.4 * overlay[mask] + 0.6 * 255 * color
        x0, y0, x1, y1 = box
        ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color=color, linewidth=2))
        ax.text(x0, y0, f'{coco_names[label]} {score:.2f}', color='white', fontsize=9, bbox=dict(facecolor=color, alpha=0.8, pad=1, linewidth=0))
    ax.imshow(overlay.astype(np.uint8))
    ax.axis('off')

fig, axs = plt.subplots(1, 3, figsize=(24, 6))
for ax, image in zip(axs[:2], [dog, me]):
    with torch.no_grad():
        out = {k: v.cpu().numpy() for k, v in maskrcnn([to_tensor(image)])[0].items()}
    show_instances(ax, image, out)
axs[0].set_title('Mask R-CNN')

with torch.no_grad():
    out = {k: v.cpu().numpy() for k, v in keypointrcnn([to_tensor(me)])[0].items()}
person = out['keypoints'][0]   # most confident person: 17 keypoints (x, y, visibility)
x0, y0, x1, y1 = out['boxes'][0]
pad = 0.15 * (y1 - y0)
axs[2].imshow(me)
for a, b in skeleton:
    axs[2].plot(person[[a, b], 0], person[[a, b], 1], '-', color='lime', linewidth=2)
axs[2].scatter(person[:, 0], person[:, 1], c='red', s=15, zorder=3)
axs[2].set_xlim(x0 - pad, x1 + pad)
axs[2].set_ylim(y1 + pad, y0 - pad)
axs[2].set_title('Keypoint R-CNN (17 COCO keypoints)')
axs[2].axis('off')
savefig('segmentation_instances_keypoints')
plt.show()

## Interactive segmentation with SAM

The Segment Anything Model encodes the image once with a large ViT, then a light-weight decoder turns *prompts* (points, boxes) into masks. A single click is ambiguous (the dog's ear? its head? the whole dog?), so SAM predicts **three** masks and a predicted IoU (its own quality estimate) for each.

In [ ]:
from transformers import SamModel, SamProcessor
sam_processor = SamProcessor.from_pretrained('facebook/sam-vit-base')
sam = SamModel.from_pretrained('facebook/sam-vit-base').to(device).eval()

# encode the image once, then decode as many prompts as we like
inputs = sam_processor(dog, return_tensors='pt').to(device)
with torch.no_grad():
    image_embeddings = sam.get_image_embeddings(inputs['pixel_values'])

def sam_masks(points=None, box=None):
    inputs = sam_processor(dog, input_points=[[points]] if points else None, input_boxes=[[box]] if box else None, return_tensors='pt').to(device)
    inputs.pop('pixel_values')
    with torch.no_grad():
        out = sam(image_embeddings=image_embeddings, **inputs)
    masks = sam_processor.image_processor.post_process_masks(out.pred_masks.cpu(), inputs['original_sizes'].cpu(), inputs['reshaped_input_sizes'].cpu())[0][0]
    return masks.numpy(), out.iou_scores[0, 0].cpu().numpy()

prompts = [('click on the ear', [[600, 140]]), ('click on the lantern', [[880, 330]])]
fig, axs = plt.subplots(len(prompts), 3, figsize=(21, 5 * len(prompts)))
for row, (name, points) in zip(axs, prompts):
    masks, scores = sam_masks(points=points)
    for ax, mask, score in zip(row, masks, scores):
        overlay = dog.astype(np.float32)
        overlay[mask] = 0.4 * overlay[mask] + 0.6 * np.array([30, 144, 255])
        ax.imshow(overlay.astype(np.uint8))
        ax.scatter(*np.array(points).T, c='red', s=80, marker='*', edgecolors='white')
        ax.set_title(f'{name}: predicted IoU {score:.2f}')
        ax.axis('off')
savefig('segmentation_sam_ambiguity')
plt.show()

In [ ]:
# a box prompt is much less ambiguous
box = [380, 30, 900, 850]
masks, scores = sam_masks(box=box)
best = masks[scores.argmax()]
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
overlay = dog.astype(np.float32)
overlay[best] = 0.4 * overlay[best] + 0.6 * np.array([30, 144, 255])
ax.imshow(overlay.astype(np.uint8))
ax.add_patch(plt.Rectangle(box[:2], box[2] - box[0], box[3] - box[1], fill=False, color='red', linewidth=2))
ax.set_title(f'Box prompt: best mask, predicted IoU {scores.max():.2f}')
ax.axis('off')
plt.show()